### Log Reader

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3-coder:480b-cloud",
    temperature=0.5,
    max_tokens=250
)

# max_tokens: the maximum number of tokens the LLM can generate in its output.
# 1 token ≈ 0.75 words in English
# So 250 tokens ≈ ~ 180 words

#### Read the log file

In [ ]:
import os
from langchain.tools import tool
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

@tool
def summarize_logs() -> str:
    "Read and returns a summary of the first few lines from each .log file. DONT get into the details of them."
    log_dir = "./logs"
    db_path = "./qa_db"
    all_logs = []
    for file in os.listdir(log_dir):
        with open(os.path.join(log_dir, file)) as f: 
        #This line opens each file inside the logs directory, gives you a file handle f, and ensures the file is closed automatically after reading.
            all_logs.extend([f"{file}: {line.strip()}" for line in f.readlines()[:50]])
            #This line reads the first 50 lines of the file, removes extra whitespace, prefixes each line with the filename, and appends all these processed log lines into the all_logs list.
    summary = "\n".join(all_logs) or "No logs found"
    #ToDo
    if not(os.path.exists(db_path) and os.path.isdir(db_path)):
        # spliting
        splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
        chunks = splitter.create_documents([summary])
        
        # embedding
        embedding = OllamaEmbeddings(model="nomic-embed-text")
        db = Chroma.from_documents(chunks, embedding, persist_directory=db_path)
        db.persist()
        summary += "\n\n[Embedding done and DB Created]"
    else:
        summary += "\n\n[Embedding skipped:DB already Created]"
    return summary

#### The code checks whether a vector database already exists.
- If it doesn’t:
    - Split the summary text
    - Create embeddings
    - Build a Chroma DB
    - Save it
- Add a note that embedding was done
    - If it already exists:
    - Skip everything
- Add a note that DB already exists

In [ ]:
from langchain_classic.agents import initialize_agent, AgentType


@tool
def add_numbers(a: int, b: int) -> int:
    "Adding two given numbers and returns a result"
    return int(a) + int(b)

@tool
def substract_numbers(a: int, b: int) -> int:
    "Subtract two given numbers and returns a result"
    return int(a) - int(b)

@tool
def multiply_numbers(a: int, b: int) -> int:
    "Multiply two given numbers and returns a result"
    return int(a) * int(b)

tools = [add_numbers, substract_numbers, multiply_numbers, summarize_logs]

# Creating an AI Agent
agent = initialize_agent(
    tools= tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)
# handle_parsing_errors = True tells LangChain to gracefully recover when the LLM returns bad or unstructured output instead of crashing the agent.

In [ ]:
tool_by_name = {tool.name: tool for tool in tools}
tool_by_name

In [ ]:
query = "What are the warnings in the application"
result = agent.run(query)
print(result)